# Translate PPMI LEDD Concomitant_Medication_Log into structured data
- Clean, parse, and standardize free-text medication records

In [1]:
import pandas as pd
import spacy
import re 
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz
import numpy as np
import unicodedata 
from spacy.matcher import PhraseMatcher

In [2]:
dir = '/Users/emudr/Desktop/data/PPMI_LEDD_Project/'
outdir = '/Users/emudr/Desktop/'
file = 'LEDD_Concomitant_Medication_Log.csv'
df = pd.read_csv(dir+file)
df = df.drop(columns=['PAG_NAME', 'SEQNO', 'LEDDOSSTR', 'LAST_UPDATE']) # Remove unneccessary columns

In [3]:
# Dictionary of substring replacements
replacements = {
    "SINEMET 250100": "SINEMET 100/250",
    "SINEMET CR 250199": "SINEMET CR 199/250",
    "RYTARY 61.75/2445 MG": "RYTARY 61.75/245 MG",
    "CARBIDOPA LEVODOPA (25-1000": "CARBIDOPA LEVODOPA (25-100)",
    "CARBIDOPA LEVODOPA (25/1000": "CARBIDOPA LEVODOPA (25-100)",
    "CARBIDOPA/LEVODOPA ER (25/1000" : "CARBIDOPA LEVODOPA (25-100) ER",
    "CARBIDOPA LEVODOPA ER(25/1000": "CARBIDOPA LEVODOPA (25-100) ER",
    "LEVODOPA/BENSERAZID 100725": "LEVODOPA/BENSERAZID 100",
    "LEVODOPA BENSERAZID 100725": "LEVODOPA/BENSERAZID 100", 
    "CARBIDOPA LEVODOPA 25-10" : "CARBIDOPA LEVODOPA 25-100",
    "CARBIDOPA-LEVODOPA 25-10" :"CARBIDOPA LEVODOPA 25-100",
    "STALEVO 100\\25\\2000" : "STALEVO 100/25/200", # FIXME dont need 
    "RYTARY (36-25-145)" : "RYTARY (36.25-145)", 
    "LEVODOPA CARBIDOPA  (20+5) INTESTINAL GEL" : "Duodopa (20+5)", 
    "LEVODOPA/CARBIDOPA INTESTINAL GEL INFUSION" : "Duodopa", 
    "PRAMIPEXOLE O,5 MG" : "PRAMIPEXOLE O.5 MG", 
    "PRAMIPEXOLE O,5 MG" : "PRAMIPEXOLE O.5 MG", # FIXME
    "LEVODOPA 7BENSERAZID 100/25" : "LEVODOPA BENSERAZID 100/25", 
    "LEVODOPA 100 MG (CORBILTA)" , "LEVODOPA 100 MG", 
    "LEVODOPA (CORBILTA) 50 MG" , "LEVODOPA 50 MG" 
}

# Apply all replacements in one pass
df['LEDTRT_replacements'] = df['LEDTRT'].replace(replacements, regex=False)
df["LEDTRT_replacements"] = df["LEDTRT_replacements"].fillna(df["LEDTRT"])

In [4]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'/', ' ', text)
    text = re.sub(r'(?<=\d),\s*(?=\d)', '.', text) # turn commas in between numbers into decimals
    text = re.sub(r'[^a-z0-9\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return unicodedata.normalize("NFKC", str(text).strip()) # FIXME check

df["med_normalized"] = df["LEDTRT_replacements"].apply(normalize_text)

In [5]:
def extract_frequency(text):
    if pd.isna(text):
        return {
            "amount": None,
            "frequency": None,
            "clean_text": text
        }

    text = str(text).lower()
    clean_text = text

    amount = None
    frequency = None

    # --- NEW: "2 pill tid", "2 tabs bid", etc.
    match = re.search(r'(\d+)\s*(pill|pills|tab|tabs|tablet|tablets)\s*(tid|bid|qid|qd)', text)
    if match:
        amount = int(match.group(1))
        freq_map = {
            "tid": 3,
            "bid": 2,
            "qid": 4,
            "qd": 1
        }
        frequency = freq_map.get(match.group(3))
        clean_text = re.sub(r'(\d+)\s*(pill|pills|tab|tabs|tablet|tablets)\s*(tid|bid|qid|qd)', '', clean_text)

    # --- x 8
    elif re.search(r'\bx\s*(\d+)', text):
        match = re.search(r'\bx\s*(\d+)', text)
        amount = 1
        frequency = int(match.group(1))
        clean_text = re.sub(r'\bx\s*\d+', '', clean_text)

    # --- "8 tabs day"
    elif re.search(r'(\d+)\s*(tabs?|tablets?)?\s*(per\s*)?(day|daily)', text):
        match = re.search(r'(\d+)\s*(tabs?|tablets?)?\s*(per\s*)?(day|daily)', text)
        amount = int(match.group(1))
        frequency = 1
        clean_text = re.sub(r'(\d+)\s*(tabs?|tablets?)?\s*(per\s*)?(day|daily)', '', clean_text)

    # --- "24 hours"
    elif re.search(r'(\d+)\s*hours?', text):
        match = re.search(r'(\d+)\s*hours?', text)
        hours = int(match.group(1))
        if hours == 24:
            amount = 1
            frequency = 1
            clean_text = re.sub(r'\d+\s*hours?', '', clean_text)

    # --- standalone abbreviations
    if re.search(r'\btid\b', text) and frequency is None:
        amount = amount or 1
        frequency = 3
        clean_text = re.sub(r'\btid\b', '', clean_text)

    if re.search(r'\bbid\b', text) and frequency is None:
        amount = amount or 1
        frequency = 2
        clean_text = re.sub(r'\bbid\b', '', clean_text)

    if re.search(r'\bqid\b', text) and frequency is None:
        amount = amount or 1
        frequency = 4
        clean_text = re.sub(r'\bqid\b', '', clean_text)

    if re.search(r'\bqd\b|\bdaily\b', text) and frequency is None:
        amount = amount or 1
        frequency = 1
        clean_text = re.sub(r'\bqd\b|\bdaily\b', '', clean_text)

    # --- cleanup
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return {
        "amount": amount,
        "frequency": frequency,
        "med_no_frequency": clean_text
    }
    
freq_info = df["med_normalized"].apply(extract_frequency).apply(pd.Series)
df = pd.concat([df, freq_info], axis=1)



In [6]:
def extract_and_remove_doses(text):
    if pd.isna(text):
        return {
            "doses": [],
            "med_no_dosages": text
        }

    text = str(text).lower()

    # extract all numbers
    nums = re.findall(r'\d+\.?\d*', text)
    doses = sorted([float(n) for n in nums])

    # remove numbers from text
    clean_text = re.sub(r'\d+\.?\d*', '', text)

    # clean formatting artifacts
    clean_text = re.sub(r'[/\-]+', ' ', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text)
    clean_text = re.sub(r'\bmg\.?\b', '', clean_text).strip()

    result = {
        "doses": doses,
        "med_no_dosages": clean_text
    }

    return result
    
dose_info = df["med_no_frequency"].apply(extract_and_remove_doses).apply(pd.Series)
df = pd.concat([df, dose_info], axis=1)



In [7]:
unique_vals = (
    pd.Series(df['med_no_dosages'].dropna().unique())
    .sort_values()
    .reset_index(drop=True)
)

unique_vals.to_csv(outdir + 'unique_values.csv', index=False)

In [8]:
#### Drug release formula 
er_terms = ["cr","er","xr","xl","sr","dr","la","sa","tr","lp", "lib", "modified release", "reseased lib", "depot", "lt", "odt",
              "extended release","prolonged/release", "prolonged release", "sustained/release",
              "sustained release","retard","slow release", "ret.", "ret"]
ir_terms = ["ir"]

# escape terms for regex
er_pattern = r'\b(?:' + '|'.join(map(re.escape, er_terms)) + r')\b'
ir_pattern = r'\b(?:' + '|'.join(map(re.escape, ir_terms)) + r')\b'


def extract_and_remove_formulation(text):
    if pd.isna(text):
        return {
            "clean_text": text,
            "release_type": None
        }

    text = str(text).lower()

    found_er = re.findall(er_pattern, text)
    found_ir = re.findall(ir_pattern, text)

    # determine type
    release_type = None
    if found_er:
        release_type = "ER"
    elif found_ir:
        release_type = "IR"

    # remove both ER + IR terms
    clean_text = re.sub(er_pattern, '', text)
    clean_text = re.sub(ir_pattern, '', clean_text)

    # clean formatting
    clean_text = re.sub(r'[/\-]+', ' ', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    clean_text = clean_text.rstrip()
    
    return {
        "clean_text": clean_text,
        "release_type": release_type
    }

form_df = df["med_no_dosages"].apply(extract_and_remove_formulation).apply(pd.Series)
df["med_no_formulation"] = form_df["clean_text"]
df["release_type"] = form_df["release_type"]


In [2]:
canonical_meds = [
    "amantadine",
    "apokyn",
    "apomorphine",
    "azilect",
    "azilect rasagiline",
    "carbidopa levodopa entacapone",
    "carbidopa levodopa",
    "carbidopa levodopa sinemet",
    "clarium",
    "co carledopa",
    "comtan",
    "dopicar", 
    "duodopa",
    "eldepryl",
    "entacapone",
    "osmolex amantadine",
    "gocovri amantadine", 
    "gocovri",
    "inbrija",
    "isicom",
    "istradefylline", 
    "kynmobi",
    "levocomp",
    "levodopa",
    "levodopa benserazide",
    "madopar",
    "mirapex",
    "nacom",
    "neupro",
    "opicapona",
    "osmolex",
    "piribedil",
    "pk merz",
    "pramipexole",
    "pramipexole mirapex",
    "rasagiline",
    "requip",
    "ropinirole",
    "rotigotine",
    "ongentys",
    "rytary",
    "rytary carbidopa levodopa", 
    "safinamide",
    "selegiline",
    "sifrol",
    "sinemet",
    "sinemet carbidopa levodopa",
    "sinemet plus",
    "stalevo",
    "tolcapone",
    "trihexyphenidyl",
    "xadago"
]

import pandas as pd

canonical_meds = sorted(canonical_meds)
df = pd.DataFrame(canonical_meds, columns=["common_meds"])

df.to_csv("/Users/emudr/Desktop/common_meds.csv", index=False)
janetl


NameError: name 'janetl' is not defined

In [10]:
nlp = spacy.load("en_core_web_sm") 
matcher = PhraseMatcher(nlp.vocab) 
patterns = [nlp.make_doc(med) for med in canonical_meds] 
matcher.add("MEDICATION", patterns) 

def extract_medication_final(text, nlp, matcher, threshold=80):
    if pd.isna(text):
        return []

    text = str(text).lower()

    if "stalevo" in text.lower(): # FIXME make own function
        text = "carbidopa levodopa entacapone"
    
    doc = nlp(text)
    all_spans = []

    # -------------------------------
    # 1. EXACT MATCHES
    # -------------------------------
    matches = matcher(doc)
    for _, start, end in matches:
        all_spans.append((start, end, doc[start:end].text))

    # -------------------------------
    # 2. FUZZY MATCHING (token + ngrams)
    # -------------------------------
    tokens = [t.text for t in doc]

    def get_ngrams(tokens, n):
        return [(i, i+n, " ".join(tokens[i:i+n])) 
                for i in range(len(tokens)-n+1)]

    candidates = (
        [(i, i+1, tokens[i]) for i in range(len(tokens))] +
        get_ngrams(tokens, 2) +
        get_ngrams(tokens, 3)
    )

    for start, end, cand in candidates:
        match, score, _ = process.extractOne(
            cand,
            canonical_meds,
            scorer=fuzz.token_sort_ratio
        )

        if score >= threshold:
            all_spans.append((start, end, match))

    # -------------------------------
    # 3. KEEP LONGEST NON-OVERLAPPING
    # -------------------------------
    # sort by start, then longest first
    all_spans = sorted(all_spans, key=lambda x: (x[0], -(x[1] - x[0])))

    final_spans = []
    occupied = set()

    for start, end, text in all_spans:
        span_tokens = set(range(start, end))

        if span_tokens & occupied:
            continue

        final_spans.append(text)
        occupied.update(span_tokens)

    return list(set(final_spans))

df["medications"] = df["med_no_formulation"].apply(lambda x: extract_medication_final(x, nlp, matcher))
df["medications"] = df["medications"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
df["medications"] = df["medications"].fillna(df["med_no_formulation"])
df.to_csv(outdir+ 'check10.csv')


In [11]:
manual_map = { 
    # other 
    "carb levo": "carbidopa levodopa",
    "ropirinol": "ropinirole",
    "tasmar": "tolcapone",
    "l dopa": "levodopa",
    "benserazida": "levodopa",
    "dopa pump" : "levodopa",
    'neuropatch' : "rotigotine",

    # Amantadine
    "amantadine": "amantadine",
    "gocovri amantadine" : "gocovri", 
    "osmolex amantadine" : "osmolex",
    "gocovri": "gocovri",
    "osmolex": "osmolex",
    "pk merz": "amantadine",

    # Apomorphine
    "apomorphine": "apomporphine",
    "apokyn": "apomporphine",

    # Carbidopa/Levodopa + Entacapone
    "carbidopa levodopa entacapone": "carbidopa levodopa entacapone",
    "stalevo": "carbidopa levodopa entacapone",

    # Carbidopa / Levodopa
    "sinemet": "carbidopa/levodopa",
    "carbidopa levodopa": "carbidopa/levodopa",
    "sinemet carbidopa levodopa": "carbidopa/levodopa",
    "carbidopa levodopa sinemet": "carbidopa/levodopa",
    "nacom": "carbidopa/levodopa",
    "levocomp": "carbidopa/levodopa",
    "sinemet plus": "carbidopa/levodopa",
    "dopicar": "carbidopa/levodopa",
    "isicom": "carbidopa/levodopa",

    # Controlled-release levodopa
    "rytary": "controlled-release levodopa",
    "rytary carbidopa levodopa": "controlled-release levodopa",
    "co carledopa": "controlled-release levodopa",

    # COMT inhibitors
    "entacapone": "entacapone",
    "comtan": "entacapone",

    # Inhaled levodopa
    "inbrija": "inbrija",

    # Adenosine A2A antagonist
    "istradefylline": "istradefylline",

    # Levodopa / Benserazide
    "levodopa": "levodopa",
    "madopar": "levodopa",
    "levodopa benserazide": "levodopa",
    "duodopa" : "duodopa", 

    # Opicapone
    "opicapona": "opicapone",
    "ongentys": "opicapone",

    # Other
    "trihexyphenidyl": "trihexyphenidyl",

    # Piribedil
    "clarium": "piribedil",
    "piribedil": "piribedil",

    # Pramipexole
    "pramipexole": "pramipexole",
    "pramipexole mirapex": "pramipexole",
    "mirapex": "pramipexole",
    "sifrol": "pramipexole",

    # Rasagiline
    "azilect": "rasagiline",
    "azilect rasagiline": "rasagiline",
    "rasagiline": "rasagiline",

    # Ropinirole
    "ropinirole": "ropinirole",
    "requip": "ropinirole",

    # Rotigotine
    "rotigotine": "rotigotine",
    "neupro": "rotigotine",

    # Safinamide
    "xadago": "safinamide",
    "safinamide": "safinamide",

    # Selegiline
    "selegiline": "selegiline",
    "eldepryl": "selegiline",

    # Tolcapone
    "tolcapone": "tolcapone",
}


df["medications_mapped"] = df["medications"].map(manual_map).fillna(df['medications'])
print(df)

cols = [
    "LEDTRT",
    "med_normalized",
    "med_no_dosages",
    "med_no_formulation",
    "doses",
    "medications",
    "medications_mapped", 
    "release_type"
]

df = df[cols]
df.to_csv(outdir + '3.csv')

                             LEDTRT                 med_normalized  \
0                    SINEMET 25/100                 sinemet 25 100   
1                        SELEGILINE                     selegiline   
2                        AMANTADINE                     amantadine   
3                    SINEMET 25/100                 sinemet 25 100   
4     ROTIGOTINE TRANSDERMAL SYSTEM  rotigotine transdermal system   
...                             ...                            ...   
6123          Carbidopa/Levodopa ER          carbidopa levodopa er   
6124                     Selegiline                     selegiline   
6125          Carbidopa/Levodopa IR          carbidopa levodopa ir   
6126          Carbidopa/Levodopa IR          carbidopa levodopa ir   
6127           Azilect (Rasagiline)             azilect rasagiline   

                   med_no_frequency                 med_no_dosages  \
0                    sinemet 25 100                        sinemet   
1                  

In [12]:
### FIX DOSAGES 


def format_dosages(row):

    text = row["medications_mapped"]
    nums = row["doses"]

    if pd.isna(text):
        return {
            "dose_1": None,
            "dose_2": None,
            "dose_3": None
        }

    text = str(text).lower()

    # --- handle missing doses safely
    if not isinstance(nums, list):
        nums = []

    # --- exact matches
    is_stalevo = text == "carbidopa levodopa entacapone"
    is_levodopa = text == "levodopa"
    is_amantadine = text == "amantadine"

    # --- stalevo: remove duplicates
    if is_stalevo:
        nums = list(dict.fromkeys(nums))

    # --- assign doses
    if (is_stalevo or is_levodopa or is_amantadine) and len(nums) == 1:
        dose_1 = None
        dose_2 = nums[0]
        dose_3 = None
    else:
        dose_1 = nums[0] if len(nums) > 0 else None
        dose_2 = nums[1] if len(nums) > 1 else None
        dose_3 = nums[2] if len(nums) > 2 else None

    return {
        "dose_1": dose_1,
        "dose_2": dose_2,
        "dose_3": dose_3
    }


dose_split = df.apply(format_dosages, axis=1).apply(pd.Series)
df = pd.concat([df, dose_split], axis=1)




## print(df["medications_mapped"].isna().sum())

In [13]:
manual_map = { 
    "carbidopa/levodopa",
    "ropinirole",
    "tolcapone",
    "levodopa",
    "rotigotine",
    "amantadine",
    "gocovri", 
    "osmolex",
    "gocovri",
    "apomporphine",
    "apomporphine",
    "carbidopa levodopa entacapone",
    "controlled-release levodopa",
    "entacapone",
    "inbrija",
    "istradefylline",
    "opicapone",
    "trihexyphenidyl"
    "piribedil",
    "pramipexole",
    "rasagiline",
    "safinamide",
    "selegiline"}


dose_cols = ["dose_1", "dose_2", "dose_3"]


result = (
    df[df["medications_mapped"].isin(manual_map)]
    .dropna(subset=["medications_mapped"])
    .loc[:, ["medications_mapped"] + dose_cols]
    .drop_duplicates()   # <-- THIS is the key part
    .sort_values(["medications_mapped"] + dose_cols)
)

print(result)

result.to_csv(outdir + "unique_doses_by_med.csv", index=False)

     medications_mapped  dose_1  dose_2  dose_3
1408         amantadine     NaN     9.0     NaN
2128         amantadine     NaN   100.0     NaN
4033         amantadine     NaN   150.0     NaN
2            amantadine     NaN     NaN     NaN
1435       apomporphine     NaN     NaN     NaN
...                 ...     ...     ...     ...
4            rotigotine     NaN     NaN     NaN
4833         safinamide    50.0     NaN     NaN
640          safinamide     NaN     NaN     NaN
1            selegiline     NaN     NaN     NaN
2595          tolcapone     NaN     NaN     NaN

[108 rows x 4 columns]


In [14]:

# load reference file
ref = pd.read_csv(dir + "med_rx_dosages.csv")

dose_cols = ["dose_1", "dose_2", "dose_3"]

# --- keep only relevant columns
df_sub = df[["LEDTRT", "medications_mapped"] + dose_cols].copy()
ref_sub = ref[["medications_mapped"] + dose_cols].copy()

# --- drop rows where ALL doses are null
df_sub = df_sub.dropna(subset=dose_cols, how="all")

# --- drop duplicates
df_sub = df_sub.drop_duplicates()
ref_sub = ref_sub.drop_duplicates()

# --- merge to find mismatches
merged = df_sub.merge(
    ref_sub,
    on=["medications_mapped"] + dose_cols,
    how="left",
    indicator=True
)


# --- rows in df NOT in reference
mismatches = merged[merged["_merge"] == "left_only"]

# --- keep clean output
mismatches = mismatches[["LEDTRT", "medications_mapped"] + dose_cols]

# --- OPTIONAL: only meds that exist in reference
mismatches = mismatches[
    mismatches["medications_mapped"].isin(ref_sub["medications_mapped"])
]

# --- 🔥 sort alphabetically + by dose
mismatches = mismatches.sort_values(
    by=["medications_mapped", "dose_1", "dose_2", "dose_3"]
)

# --- save
mismatches.to_csv(outdir + "Xdose_mismatches.csv", index=False)
print(mismatches)

                                   LEDTRT           medications_mapped  \
101                          AMANTADINE09                   amantadine   
144              CARBIDOPA/LEVODOPA 2/100           carbidopa/levodopa   
261                          SINEMET 12.5           carbidopa/levodopa   
181  CARBIDOPA/LEVODOPA 25/100 MG  25/100           carbidopa/levodopa   
260                      SINEMET PLUS 100           carbidopa/levodopa   
279                     SINEMET CR 250199           carbidopa/levodopa   
50                             RYTARY 145  controlled-release levodopa   
428                             RYTARY195  controlled-release levodopa   
367                           MADOPAR 200                     levodopa   
339                       PRAMIPEXOL 0,35                  pramipexole   
340                        PRAMIPEXOL 0,7                  pramipexole   
347                       SIFROL 1, 57 MG                  pramipexole   
63      PRAMIPEXOLE (MIRAPEX) .125 MG 